# Burn Simulation

In [1]:
import plotly.express as px
import pandas as pd
import numpy as np


import plotly.figure_factory as ff
import plotly.graph_objects as go
from plotly.express.colors import qualitative as pallettes

from copy import deepcopy

In [2]:
def profile2(theta):
    r = 10 / (np.cos(np.pi/4 - (theta % (np.pi/2))))
    return r

In [3]:
def profile1(theta):
    a = 10 # offset
    b = 5 # amplitude
    c = 7 # number of lobes
    r = a + b * np.cos(c*theta)
    return r

In [4]:
# Convert between Cartesian/Polar coordinates

def cart2pol(x, y):
    rho = np.sqrt(x**2 + y**2)
    phi = np.arctan2(y, x)
    return(rho, phi)

def pol2cart(rho, phi):
    x = rho * np.cos(phi)
    y = rho * np.sin(phi)
    return(x, y)

def magn(x, y):
    # magnitude of vector
    return np.sqrt(x**2 + y**2)

def rad2deg(rad):
    # convert radians to degrees
    return rad * 180 / np.pi

def circumference(X,Y):
    with np.errstate(divide='ignore', invalid='ignore'):
        dx = np.diff(X)
        dy = np.diff(Y)
    dxdy = np.sqrt(dx**2 + dy**2)
    C = np.sum(dxdy)
    return C 

In [5]:
def normals(X, Y, ):
    # Calculates normals of the curve

    Gy = np.gradient(Y, X)

    #Check if any gradients got inf or nan
    #assert sum(np.isnan(Gy)) == 0

    #Gx = np.gradient(X, Y) # this should give us an array of 1s, since the gradient of X with respect to itself is 1.

    Gx = np.ones_like(Gy) # not sure this is correct btw.
    # Gx are all 1s assumes the curve Y is a function of X, which is an axis with uniform spacing.
    # But in our case, the X and Y are both functions of theta, since they are generated from a polar coordinate system,
    # So, the assumption may not hold, and the normals may not be accurate. 
    # And that is probably why in some regions the normals don't look perpendicular to the curve in the plot.

    # Calculate normals
    L = magn(Gx, Gy)
    Nx, Ny = -Gy/L, Gx/L 
    xy  = np.array([X,Y])
    NxNy = np.array([Nx,Ny])

    # Ensure all normal vectors point in the same 
    # direction (Relative to origin) 
    dot_products = np.sum(xy * NxNy, axis=0)

    scaling_factor_S = np.sign(dot_products)

    S_reshaped = scaling_factor_S[np.newaxis, :]

    N_consistent = NxNy * S_reshaped 
    Nx, Ny =  N_consistent
    return Nx, Ny

In [6]:

def arc_lens(X,Y):
    """
    Calculate lengths of curve segments defined by points (X,Y).

    Args:
        X (_type_): _description_
        Y (_type_): _description_

    Returns:
        _type_: _description_
    """

    XY = np.stack((X,Y), axis=1)
    XY = np.concatenate((XY,XY[0:1,:]), axis=0)

    diff = np.diff(XY, axis=0)
    Lens = np.sqrt(np.sum(diff ** 2, axis=1))
    return Lens

In [1]:
def shape(item):
    if hasattr(item, '__getitem__'):
        if hasattr(item, 'shape'):
            return item.shape
        else:
            if hasattr(item, '__len__'):
                return len(item)
            else:
                raise ValueError("Item has __getitem__ but no shape or length.")
    else:
        return item

In [3]:
s = 0

isSubscriptable = lambda obj: hasattr(obj, '__getitem__')

def describe(subst):
    fmt = "{n} {s}: {t}"
    summ = [fmt.format(n=k, t=type(v), s= shape(v) ) for k,v in subst.items() ]
    print(*summ, sep='\n')

def clip(vars, indcs = slice(None)):

    rooster = vars.split(", ")
    #print(rooster)
    thevars = {k: globals()[k] for k in rooster}

    subst = {k: v[indcs] if isSubscriptable(v) else v  for k, v in thevars.items()}
    describe(subst)
    return subst






In [5]:
describe(globals())

__name__ 8: <class 'str'>
__doc__ 121: <class 'str'>
__package__ None: <class 'NoneType'>
__loader__ None: <class 'NoneType'>
__spec__ None: <class 'NoneType'>
__builtin__ <module 'builtins' (built-in)>: <class 'module'>
__builtins__ <module 'builtins' (built-in)>: <class 'module'>
_ih 6: <class 'list'>
_oh 2: <class 'dict'>
_dh 1: <class 'list'>
In 6: <class 'list'>
Out 2: <class 'dict'>
get_ipython <bound method InteractiveShell.get_ipython of <ipykernel.zmqshell.ZMQInteractiveShell object at 0x77168967b210>>: <class 'method'>
exit <IPython.core.autocall.ZMQExitAutocall object at 0x77168968f050>: <class 'IPython.core.autocall.ZMQExitAutocall'>
quit <IPython.core.autocall.ZMQExitAutocall object at 0x77168968f050>: <class 'IPython.core.autocall.ZMQExitAutocall'>
open <function open at 0x77168aa54720>: <class 'function'>
_ 2: <class 'str'>
__ 2: <class 'str'>
___ 2: <class 'str'>
__vsc_ipynb_file__ 58: <class 'str'>
__DW_SCOPE__ 2: <class 'dict'>
_i 53: <class 'str'>
_ii 479: <class 'st

In [8]:
#simulation points
n = 1000 

# Theta (radians). To avoid /div0 start from 1 
T = np.linspace(1, np.pi*2 +1 , n)

# Theta degrees
TD = rad2deg(T)

In [9]:
# Calculate Radius for each Theta
R = profile1(T)

## Working in cartesian


In [395]:
X, Y = pol2cart(R, T)

Calculate the gradient (which in 1d case is just derivative)

In [396]:
Gy = np.gradient(Y, X)

Check if any gradients got inf or nan

In [397]:
sum(np.isnan(Gy))

0

In [398]:
infs = np.argwhere(Gy == -np.inf)
infs

array([], shape=(0, 1), dtype=int64)

In [399]:
infs = np.argwhere(Gy == np.inf)
infs

array([], shape=(0, 1), dtype=int64)

Optionally: set them to 0? 

In [ ]:
Gy[infs] = 0

In [400]:
px.line(Gy)

In [406]:
Gx = np.ones_like(Gy)
#Gx = np.gradient(X,Y)

In [407]:
px.line(Gx)

In [408]:
L = magn(Gx, Gy)
L.shape


(1000,)

In [405]:
px.line(L)

In [409]:
px.line(L)

Calculate normals

In [ ]:
Nx, Ny = -Gy/L, Gx/L 

In [ ]:
xy  = np.array([X,Y])

In [ ]:
NxNy = np.array([Nx,Ny])

In [ ]:
NxNy.shape

Ensure all normal vectors point in the same direction 
(Relative to origin) 

In [ ]:
dot_products = np.sum(xy * NxNy, axis=0)
dot_products.shape

In [ ]:
scaling_factor_S = np.sign(dot_products)

# 3. Reshape S to be an N x 1 array to multiply correctly with N_initial (N x 2)
#    np.newaxis expands the dimension: [s1, s2, ...] -> [[s1], [s2], ...]
S_reshaped = scaling_factor_S[np.newaxis, :]

In [ ]:
S_reshaped.shape

In [ ]:
N_consistent = NxNy * S_reshaped 
N_consistent.shape

In [ ]:
Nx, Ny =  N_consistent

In [ ]:
Nx.shape

Calculate the new points after iteration

In [ ]:
# Iteration step size
d = -.1

X1, Y1 =  X - d * Nx , Y - d * Ny

In [ ]:
# region of interest
roi = {'x':(-16,16), 'y':(-16,16)}
#roi = {'x':(0.5, 2.), 'y':(4.5, 6.5)} # for the 7-lobe shape
print(f"zooming to x: {roi['x']}, y: {roi['y']}")

In [ ]:

fig = px.line(x = X, y = Y,width=1000, height=1000)
fig.add_trace(go.Scatter(x=X1 , y=Y1, mode='lines', name='next step'))
fig.update_xaxes(range=roi['x'])
fig.update_yaxes(range=roi['y'])
fig.show()

Calcualte circumference

In [ ]:


circumference(X,Y)

## Problem

In [ ]:
# Larger iteration step size
d = .3

# Offset
Ox, Oy =  d * Nx , d * Ny
# End
X1, Y1 =  X + Ox,  Y + Oy 

In [ ]:

fig = px.line(x = X, y = Y,width=800, height=800, )
fig.add_trace(go.Scatter(x=X1 , y=Y1, mode='lines', name='next step'))
fig.update_xaxes(range=roi['x'])
fig.update_yaxes(range=roi['y'])
fig.show()

If I increase iteration step size, then in sharp corners of the shape we'll inevitably get normals crossing to the other side of the shape, thereby creating these "swallow tail" patterns.  
Same goes for any further iterations.  
Naturally, circumference calculation is also affected by this.

I need to cut these tails off.

## Vector intersections

Observing the plot, I see that the normals producing the tail are the ones that intersect. These are the ones I need to remove



In [ ]:
fig = ff.create_quiver(X,Y, Ox,Oy,
                        #x, y, u, v,
                       scale=1, 
                       arrow_scale=.05,
                       name='offset',
                       line_width=3, 
                       angle=np.pi/6,
                     #  width=800, height=800)
)

fig.add_trace(go.Scatter (x=X, y=Y,mode='lines', name='current'))
fig.add_trace(go.Scatter (x=X1, y=Y1,mode='lines',  name='next step'))

fig.update_layout(  width=800, height=800)
fig.update_xaxes(range=roi['x'])
fig.update_yaxes(range=roi['y'])
# Add points to figure
fig.show()

Solution Approach:  
Drop all normals that are adjacent and intersect within distance d (the step size) from their origin.  


**Summary of the function:**  

Given: an array of size n of 2d vectors. Each vector has an origin X,Y and direction Nx, Ny.  
Order of vectors matters. For each pair of vectors adjacent within the array, find their intersection point.   
No need to check for intersections of all possible pairs of vectors. Just the adjacent ones.  
Only intersections in the positive direction of vectors matter.  

In [ ]:
def adjacent_intersections(X, Y, Nx, Ny, forward_only=True, eps=1e-12):
    """Compute intersections for adjacent 2D rays (vectorized)."""
    X = np.asarray(X).ravel()
    Y = np.asarray(Y).ravel()
    Nx = np.asarray(Nx).ravel()
    Ny = np.asarray(Ny).ravel()
    if not (X.size == Y.size == Nx.size == Ny.size):
        raise ValueError('All inputs must have same length')
    # Stack as (n,2) arrays and form adjacent pairs (n-1)
    c = np.stack((X, Y), axis=1)  # (n,2)
    v = np.stack((Nx, Ny), axis=1)
    c_i = c[:-1]; c_j = c[1:]
    v_i = v[:-1]; v_j = v[1:]
    C = c_j - c_i
    # 2D cross product (scalar) for arrays of shape (m,2)
    def cross2(a, b):
        return a[:, 0] * b[:, 1] - a[:, 1] * b[:, 0]
    den = cross2(v_i, v_j)
    num_ti = cross2(C, v_j)
    num_tj = cross2(C, v_i)
    # safe division with NumPy (will produce inf/nan where den==0)
    with np.errstate(divide='ignore', invalid='ignore'):
        ti = num_ti / den
        tj = num_tj / den
    # validity mask: non-parallel and finite
    valid = np.isfinite(den) & (np.abs(den) > eps)

    if forward_only:
        valid &= (ti > 0) & (tj > 0) 

    # Intersection points computed from ray i: p = c_i + ti * v_i
    Px = c_i[:, 0] + ti * v_i[:, 0]
    Py = c_i[:, 1] + ti * v_i[:, 1]
    # Mask invalid entries as NaN for clarity
    invalid = ~valid
    if invalid.any():
        Px = Px.astype(float)
        Py = Py.astype(float)
        ti = ti.astype(float)
        tj = tj.astype(float)
        # Px[invalid] = np.nan
        # Py[invalid] = np.nan
        # ti[invalid] = np.nan
        # tj[invalid] = np.nan
    return Px, Py, ti, tj, valid


Px, Py, ti, tj, valid = adjacent_intersections(X, Y, Nx, Ny, forward_only=True)

In [ ]:
# filter which drops points that fall within d from previous line

def myfilter(ti,tj,d):
    a= np.append(ti, np.array(True))
    b = np.append(np.array(True), tj)
    flt = (a <= 2.2*np.abs(d) ) & (b <= 2.2*np.abs(d) )
    return flt

In [ ]:
filt = myfilter(ti,tj,d) & np.append(valid, False)

Xn = X1[~filt]
Yn = Y1[~filt]

In [ ]:
fig = ff.create_quiver(X,Y, Ox, Oy, 
                        #x, y, u, v,
                       scale=1, 
                       arrow_scale=.05,
                       name='offset',
                       line_width=3, 
                       angle=np.pi/6,
                     #  width=800, height=800)
)

fig.add_trace(go.Scatter (x=X, y=Y,mode='lines', name='current step'))
fig.add_trace(go.Scatter (x=Px, y=Py,mode='markers',  name='intersections'))
fig.add_trace(go.Scatter (x=Xn, y=Yn,mode='lines',  name='next step'))

fig.update_layout(  width=800, height=800)
fig.update_xaxes(range=roi['x'])
fig.update_yaxes(range=roi['y'])
# Add points to figure
fig.show()

There is some progress: The vectors situated in the sharpest part of the curve got filtered out.  

However intersecting vectors still remain. 
Those are the vectors that fail our initial assumption that only adjacent intersecting vectors need to be dropped.  

The approach needs to be modified.  


## Modified approach

It is necessary to check for intersections of not only the adjacent vectors, but more.  
Thankfully, no need to check ALL the possible intersections, but a relatively small amount of vectors  
within some sliding window. This will keep the performance sane.

Generally the procedure should be:  
Run a window over the array, with defined length and offset.  
For each offset, only check intersections of the first vector vs. all the rest of the vectors inside the window.  
All the normals pairs that have their intersections within the bounds of both normals, are to be dropped.  

The implementation of this approach is below:


In [75]:
def window_intersections(X, Y, Nx, Ny, window_size=11, step=1, tol=0.1, forward_only=True, eps=1e-12):
    """Find intersections within sliding windows.
    For each window start, test the first ray (index i=start) against all following rays in the window.
    Return compact arrays of hits (i, j, ti, tj, Px, Py)."""

    X = np.asarray(X).ravel()
    Y = np.asarray(Y).ravel()
    Nx = np.asarray(Nx).ravel()
    Ny = np.asarray(Ny).ravel()
    if not (X.size == Y.size == Nx.size == Ny.size):
        raise ValueError('All inputs must have same length')
    n = X.size
    if window_size < 2:
        raise ValueError('window_size must be >= 2')
    # Prepare arrays
    c = np.stack((X, Y), axis=1)  # (n,2)
    v = np.stack((Nx, Ny), axis=1)

    hits_i = []
    hits_j = []
    hits_ti = []
    hits_tj = []
    hits_P = []
    hits_Py = []
    classes = []
    condis = []
    condjs = []
    filt = np.zeros(n).astype(bool)
    iis = []

    # Helper cross product for arrays
    # def cross2_arr(a_x, a_y, b_x, b_y):
    #     return a_x * b_y - a_y * b_x
    # Slide window (simple Python loop over windows; per-window ops are vectorized)

    print("points to do:",  n - window_size + 1)
    
    for i in range(0, n - window_size + 1, step):
        this = i + int(window_size/2)
        c0 = c[this]            # (2,)
        v0 = v[this]            # (2,)
        c_block = c[i+1:i+window_size]    # (m,2)
        v_block = v[i+1:i+window_size]    # (m,2)
        # Cx = c_block[:,0] - c0[0]
        # Cy = c_block[:,1] - c0[1]
        C = c_block - c0
        # den = cross(v0, v_block)
        # den = cross2_arr(v0[0], v0[1], v_block[:,0], v_block[:,1])
        den = np.cross(v0,v_block)
        # numerators
        # num_ti = cross2_arr(Cx, Cy, v_block[:,0], v_block[:,1])
        num_ti = np.cross(C, v_block)
        # num_tj = cross2_arr(Cx, Cy, v0[0], v0[1])
        num_tj = np.cross(C, v0)

        with np.errstate(divide='ignore', invalid='ignore'):
            ti = num_ti / den
            tj = num_tj / den

        valid = den != 0 #& (np.abs(den) > eps)
        valid = valid & (((-3 < ti) & (ti  < tol*3)) | ((-3 < tj ) & (tj < tol*3)))

        condi = (0 < ti) & (ti < tol*.9)
        condj = (0 < tj) & (tj < tol)


        # these must be none true
        outer = all( condi== False) #| all( condj == False )

        clas = valid*1 + condi*1 + condj*1

            
        # filt[this] = (not any((condi) & (condi != condj))) #| outer
        
        filt[this] = (not any((condi) & (condi != condj))) & (not all( condi== False))

        #filt[this] = outer
        #valid &= cond
        # if not np.any(valid):
        #     continue

        # if filt.sum() > 0:
        #     print('wait', i, end='')

        # compute intersection points for valid entries
        # vi_x = v0[0]; 
        # vi_y = v0[1]
        # Px = c0[0] + ti * vi_x
        # Py = c0[1] + ti * vi_y

        P = c0 + np.stack((ti,ti), axis=1) * v0

        # append hits
        # for valid in np.nonzero(valid)[0]:
        # hits_i.append(i)
        # hits_j.append(i + 1 + int(idx_local))

        iis.append(i)
        hits_ti.append(ti[valid])
        hits_tj.append(tj[valid])
        hits_P.append(P[valid])
        classes.append(clas[valid])
        condis.append(condi[valid])
        condjs.append(condj[valid])

    res = {
            "iis":     np.array(iis),
            # hits_i: hits_i,
            # hits_j: hits_j,
            "hits_ti": np.concatenate(hits_ti), 
            "hits_tj": np.concatenate(hits_tj), 
            "hits_P":  np.concatenate(hits_P), 
            "classes": np.concatenate(classes), 
            "condis":  np.concatenate(condis), 
            "condjs":  np.concatenate(condjs), 
            "filt":    np.array(filt),
        }
    return res

    # if len(hits_i) == 0:
    #     return (np.array([], dtype=int), np.array([], dtype=int), np.array([], dtype=float),
    #             np.array([], dtype=float), np.array([], dtype=float), np.array([], dtype=float),
    #             np.array([]), np.array([]), np.array([]), np.array([]), np.array([]) )

    # return (np.array(hits_i, dtype=int), np.array(hits_j, dtype=int), np.array(hits_ti, dtype=float),
    #         np.array(hits_tj, dtype=float), np.array(hits_P, dtype=float), np.array(hits_Py, dtype=float),
    #         np.array(classes), np.array(condis), np.array(condjs), np.array(filt), np.array(iis) )

In [76]:
def curve_intersections(X, Y, Nx, Ny, window_size=11, step=1, tol=0.1, forward_only=True, eps=1e-12):
    """Find intersections within sliding windows.
    For each window start, test the first ray (index i=start) against all following rays in the window.
    Return compact arrays of hits (i, j, ti, tj, Px, Py)."""

    # I = np.asarray(I).ravel()
    X = np.asarray(X).ravel()
    Y = np.asarray(Y).ravel()
    Nx = np.asarray(Nx).ravel()
    Ny = np.asarray(Ny).ravel()
    if not (X.size == Y.size == Nx.size == Ny.size):
        raise ValueError('All inputs must have same length')
    n = X.size
    if window_size < 2:
        raise ValueError('window_size must be >= 2')
    # Prepare arrays
    c = np.stack((X, Y), axis=1)  # (n,2)
    # v = np.stack((Nx, Ny), axis=1)
    c1 = np.concatenate((c[-2:,:],c[:-2] ), axis=0)

    v = c1 - c

    hits_ti = []
    hits_tj = []
    hits_P = []
    classes = []
    condis = []
    condjs = []
    filt = np.zeros(n).astype(bool)
    iis = []

    # print("points to do:",  n - window_size + 1)
    
    for i in range(0, n - window_size + 1, step):
        this = i #+ int(window_size/2)
        c0 = c[this]            # (2,)
        v0 = v[this]            # (2,)
        c_block = c[i+2:i+window_size]    # (m,2)
        v_block = v[i+2:i+window_size]    # (m,2)

        C = c_block - c0

        den = np.cross(v0,v_block)

        num_ti = np.cross(C, v_block)

        num_tj = np.cross(C, v0)

        with np.errstate(divide='ignore', invalid='ignore'):
            ti = num_ti / den
            tj = num_tj / den

        l, u = -0.1, 1.1
        valid = den != 0 
        # valid = valid & (((l < ti) & (ti  < u)) | ((l < tj ) & (tj < u)))

        condi = (0 <= ti) & (ti < 1)
        condj = (0 <= tj) & (tj < 1)

        valid = valid & condi & condj

        # these must be none true
        #outer = all( condi== False) #| all( condj == False )

        clas = valid*1 + condi*1 * condj*1

        if any(condi & condj):
            j = i + 2 + np.min(np.where(condi & condj))
            filt[i:j] = True

        # filt[this] = (not any((condi) & (condi != condj))) & (not all( condi== False))
        #filt[this] = any(condi & condj)


        P = c0 + np.stack((ti,ti), axis=1) * v0

        iis.append(i)
        hits_ti.append(ti[valid])
        hits_tj.append(tj[valid])
        hits_P.append(P[valid])
        classes.append(clas[valid])
        condis.append(condi[valid])
        condjs.append(condj[valid])

    res = {
            "iis":     np.array(iis),
            "hits_ti": np.concatenate(hits_ti), 
            "hits_tj": np.concatenate(hits_tj), 
            "hits_P":  np.concatenate(hits_P), 
            "classes": np.concatenate(classes), 
            "condis":  np.concatenate(condis), 
            "condjs":  np.concatenate(condjs), 
            "filt":    np.array(filt),
        }
    return res


In [84]:

def step(I, X, Y, d, s):
    Nx, Ny = normals(X, Y)

    Ex, Ey =  X + d * Nx , Y + d * Ny

    res = curve_intersections(Ex, Ey, Nx, Ny, window_size=50, step=1, tol=d) # *(1+s*0.1)
    
    #res = window_intersections(Ex, Ey, Nx, Ny, window_size=50, step=1, tol=d) # *(1+s*0.1)


    iis, ti_w, tj_w, P, clss, condi, condj, filt = res.values()
    # filt is True where the points should be filtered out / dropped
    
    #Px, Py, ti, tj, valid = adjacent_intersections(Ex, Ey, Nx, Ny, forward_only=True)
    #filt =  np.concatenate([i_idx , j_idx])  #myfilter(ti,tj,d) # & np.append(valid, False)
    
    # if filt.size != 0:
    #     Ex[filt] = np.nan
    #     Ey[filt] = np.nan

    if filt.size != 0:
    #     Ex[filt] = np.nan
    #     Ey[filt] = np.nan


        X1 = Ex[~filt]
        Y1 = Ey[~filt]
        I1 =  I[~filt]


    res = { 'I':I,
        'step': s,
        # 'Theta': T, 
        # 'ThetaDeg':TD, 
        # 'Radius': R, 
        # 'Gradient': GP,
        'X': X,
        'Y': Y,
        'Nx': Nx,
        'Ny': Ny,
        'Ex': Ex,
        'Ey': Ey,
        'I1': I1,
        'X1': X1,
        'Y1': Y1,
        'filt': filt,
        'circ': circumference(X,Y),
        'iis': iis
        }
    extra =  { 'step': s,
        'Px': P[:,0],
        'Py': P[:,1],
        'ti': ti_w,
        'tj': tj_w,
        'condi': condi, 
        'condj': condj,
        'clss' : clss
        # 'valid': np.append(valid, False),
        }


    return I1, X1, Y1, res, extra


In [85]:

def dots_and_arrows(I, X, Y, Nx, Ny, I1, X1, Y1, Ex, Ey, Px, Py, d, clss, filt, **kwargs):
    fig = ff.create_quiver(X, Y, +d * Nx, +d * Ny, scale=1, arrow_scale=.05, name='offset',hovertext=I)
    fig.add_trace(go.Scatter(x=Px, y=Py, mode='markers', 
                             marker=dict(size=6, color = clss*1, symbol = 'x') , 
                             hovertext = clss,
                             name='window_hits'))
    fig.add_trace(go.Scatter(x=X1, y=Y1, mode='lines', marker=dict(size=6, color=filt*1), 
                             hovertext=I1, name='X1Y1'))
    fig.add_trace(go.Scatter(x=Ex, y=Ey, mode='lines', marker=dict(size=6, color=filt*1), 
                             hovertext=I, name='ExEy'))
    fig.update_layout(width=800, height=800)
    # fig.update_xaxes(range=roi['x'])
    # fig.update_yaxes(range=roi['y'])
    fig.show()


#dots_and_arrows(I, X,Y, Nx, Ny, Ex, Ey, Px_w, Py_w,d)


In [86]:
I1, X1, Y1, res, extra =  step(**subst)

In [87]:
# I1, X1, Y1, res, extra =  step(I1, X1, Y1, d,s)

In [88]:
tmp = res
tmp.update(extra)
# [print(k, v.shape) for k,v in tmp.items() if type(v) not in (int, float, np.float64)]
#tmp = {k: tmp[k] for k in "I, X, Y, Nx, Ny, X1, Y1, Ex, Ey, Px, Py".split(", ")}
describe(tmp)
#[print(k, v.shape) for k,v in tmp.items()]

I (1000,): <class 'numpy.ndarray'>
step 0: <class 'int'>
X (1000,): <class 'numpy.ndarray'>
Y (1000,): <class 'numpy.ndarray'>
Nx (1000,): <class 'numpy.ndarray'>
Ny (1000,): <class 'numpy.ndarray'>
Ex (1000,): <class 'numpy.ndarray'>
Ey (1000,): <class 'numpy.ndarray'>
I1 (851,): <class 'numpy.ndarray'>
X1 (851,): <class 'numpy.ndarray'>
Y1 (851,): <class 'numpy.ndarray'>
filt (1000,): <class 'numpy.ndarray'>
circ (): <class 'numpy.float64'>
iis (951,): <class 'numpy.ndarray'>
Px (41,): <class 'numpy.ndarray'>
Py (41,): <class 'numpy.ndarray'>
ti (41,): <class 'numpy.ndarray'>
tj (41,): <class 'numpy.ndarray'>
condi (41,): <class 'numpy.ndarray'>
condj (41,): <class 'numpy.ndarray'>
clss (41,): <class 'numpy.ndarray'>


In [89]:

dots_and_arrows(**tmp, d = d)

In [49]:
from Superformula.Formulas import formula1, formula2

In [50]:
# args = ('superformula', 3, 1, 0.7, 2, 0, 1, 0)

# args = ('superformula', 5, 1, -0.38, 0.38, 0, 1, 0)

# args = ('superformula', 5, 1, -0.66, 0.95, 0, 1, 1)

args = ('trapez wave', 5, 1, -0.66, 0.95, 0, 1, 1)

In [51]:
profile = formula2(*args)

In [52]:
d = .11
steps = 20
n = 1000 

T = np.linspace(0, np.pi*2 , n)
I = np.arange(len(T))
# Calculate Radius for each Theta
R = profile(T)

X, Y = pol2cart(R, T)
Nx, Ny = normals(X, Y)
X1, Y1 =  X + d * Nx , Y + d * Ny



In [53]:
# indices = slice(224,324)

indices = slice(None)

# subst = [ii[indices] for ii in [X, Y, Nx, Ny,]]

vrs = "I, X, Y, d, s"

subst = clip(vrs, indices)

I (1000,): <class 'numpy.ndarray'>
X (1000,): <class 'numpy.ndarray'>
Y (1000,): <class 'numpy.ndarray'>
d 0.11: <class 'float'>
s 0: <class 'int'>


In [54]:
describe(subst)

I (1000,): <class 'numpy.ndarray'>
X (1000,): <class 'numpy.ndarray'>
Y (1000,): <class 'numpy.ndarray'>
d 0.11: <class 'float'>
s 0: <class 'int'>


In [379]:
Lens = arc_lens(X,Y)
np.mean(Lens)

0.015770860428220373

In [380]:
px.line(Lens)

In [ ]:
# Demo: run window_intersections on current arrays and plot hits




In [ ]:


# i_idx, j_idx, ti_w, tj_w, Px, Py, clss, condi, condj, filt, iiss = window_intersections(*subst, forward_only=False, window_size=90, step=1, tol=d)
# print('window_intersections found', i_idx.size, 'hits')
# if i_idx.size > 0:
#     print('sample (i,j,ti,tj):')
#     for a,b,ta,tb in zip(i_idx[:10], j_idx[:10], ti_w[:10], tj_w[:10]):
#         print(a, b, round(ta,3), round(tb,3))
# # Plot hits on top of the offset quiver





In [ ]:
# describe(res)

In [ ]:
# dots_and_arrows(I[indices], *subst, Ex_f[~filt], Ey_f[~filt], Px, Py,d, condi == condj)

In [ ]:
# dots_and_arrows(*subst, Px, Py,d)

# Full simulation 

Following is all above logic summarized in easy to use functions.

In [344]:



def run(func, d = .011, steps = 1, n = 1000 ):
    """runs the simulation

    Args:
        func (def): function defining the curve. must be of form Radius = f(Theta)
        d (float, optional): step size. Defaults to -.3.
        steps (int, optional): simulation steps. Defaults to 1.
        n (int, optional): simulation points. Defaults to 1000.

    Returns:
        dict: dictionary of results data
    """
    
    print('Simulation step size(d):',d)
    print('Simulation Steps:', steps)
    print('Curve points (n):', n)

    # Theta (radians)
    T = np.linspace(0, np.pi*2 , n)
    I = np.arange(len(T))
    # Calculate Radius for each Theta
    R = func(T)

    X, Y = pol2cart(R, T)

    data = []
    extras = []

    for s in range(steps):
        print("step:", s, "points:", X.shape)

        I, X, Y,res, extra = step(I, X, Y, d, s)

        data.append(deepcopy(res))
        extras.append(deepcopy(extra))


    return data, extras


## Run the simulation

In [345]:
from Superformula.Formulas import formula1, formula2

In [346]:
# args = ('superformula', 3, 1, 0.7, 2, 0, 1, 0)

# args = ('superformula', 5, 1, -0.38, 0.38, 0, 1, 0)

#args = ('superformula', 5, 1, -0.66, 0.95, 0, 1, 1)

args = ('trapez wave', 5, 1, -0.66, 0.95, 0, 1, 1)

In [347]:
profile = formula2(*args)

In [363]:

d = .11
steps = 40

n = 1000

# Theta (radians). To avoid /div0 start from 1 
T = np.linspace(0, np.pi*2 , n)

R = profile(T)

Rmax = np.max(R) 


X, Y = pol2cart(R, T)
mode='lines'
fig = px.line(x=X, y=Y)
fig.update_layout(width=500 , height = 500, yaxis_range = [-Rmax, Rmax], xaxis_range = [-Rmax, Rmax])

fig.show()

In [364]:


data, extra = run(profile, d, steps, n)

Simulation step size(d): 0.11
Simulation Steps: 40
Curve points (n): 1000
step: 0 points: (1000,)
step: 1 points: (851,)
step: 2 points: (728,)
step: 3 points: (640,)
step: 4 points: (582,)
step: 5 points: (534,)
step: 6 points: (492,)
step: 7 points: (456,)
step: 8 points: (424,)
step: 9 points: (395,)
step: 10 points: (369,)
step: 11 points: (345,)
step: 12 points: (324,)
step: 13 points: (303,)
step: 14 points: (286,)
step: 15 points: (272,)
step: 16 points: (269,)
step: 17 points: (264,)
step: 18 points: (264,)
step: 19 points: (261,)
step: 20 points: (258,)
step: 21 points: (258,)
step: 22 points: (258,)
step: 23 points: (258,)
step: 24 points: (258,)
step: 25 points: (258,)
step: 26 points: (258,)
step: 27 points: (258,)
step: 28 points: (258,)
step: 29 points: (258,)
step: 30 points: (258,)
step: 31 points: (258,)
step: 32 points: (258,)
step: 33 points: (258,)
step: 34 points: (258,)
step: 35 points: (258,)
step: 36 points: (258,)
step: 37 points: (258,)
step: 38 points: (258,)

In [365]:
describe(data[1])

I (851,): <class 'numpy.ndarray'>
step 1: <class 'int'>
X (851,): <class 'numpy.ndarray'>
Y (851,): <class 'numpy.ndarray'>
Nx (851,): <class 'numpy.ndarray'>
Ny (851,): <class 'numpy.ndarray'>
Ex (851,): <class 'numpy.ndarray'>
Ey (851,): <class 'numpy.ndarray'>
I1 (728,): <class 'numpy.ndarray'>
X1 (728,): <class 'numpy.ndarray'>
Y1 (728,): <class 'numpy.ndarray'>
filt (851,): <class 'numpy.ndarray'>
circ (): <class 'numpy.float64'>
iis (802,): <class 'numpy.ndarray'>


In [366]:
def pick(dt):
    fields = "I, step, X, Y, Nx, Ny, circ".split(', ')
    toplot = {f: dt[f] for f in fields}
    return toplot

pick(data[1])

{'I': array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
         13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
         26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
         39,  40,  41,  42,  43,  44,  45,  46,  62,  63,  64,  65,  66,
         67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,  79,
         80,  81,  82,  83,  84,  85,  86,  87,  88, 104, 105, 106, 107,
        108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120,
        121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133,
        134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146,
        147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159,
        160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172,
        173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185,
        186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198,
        199, 200, 201, 202, 203, 204, 205, 206

In [367]:
df = [pd.DataFrame(pick(dt)) for dt in data]
df = pd.concat(df, ignore_index=True)
df

,I,step,X,Y,Nx,Ny,circ
0,0,0,2.000000,0.000000,0.527437,0.849594,15.770860
1,1,0,1.979941,0.012453,0.522855,0.852421,15.770860
2,2,0,1.959805,0.024654,0.513586,0.858038,15.770860
3,3,0,1.939595,0.036601,0.504235,0.863566,15.770860
4,4,0,1.919312,0.048296,0.494802,0.869006,15.770860
...,...,...,...,...,...,...,...
14214,995,39,6.288155,-0.152285,0.999729,-0.023295,39.037222
14215,996,39,6.289029,-0.110332,0.999869,-0.016198,39.037222
14216,997,39,6.289636,-0.067059,0.999960,-0.008916,39.037222
14217,998,39,6.289933,-0.027789,0.999994,-0.003433,39.037222


In [368]:
df.isna().sum()

I       0
step    0
X       0
Y       0
Nx      0
Ny      0
circ    0
dtype: int64

In [369]:
df[df.Nx.isna() | df.Ny.isna()]

,I,step,X,Y,Nx,Ny,circ


In [370]:
df = df.drop("circ", axis=1).dropna(axis=0)

In [371]:
df

,I,step,X,Y,Nx,Ny
0,0,0,2.000000,0.000000,0.527437,0.849594
1,1,0,1.979941,0.012453,0.522855,0.852421
2,2,0,1.959805,0.024654,0.513586,0.858038
3,3,0,1.939595,0.036601,0.504235,0.863566
4,4,0,1.919312,0.048296,0.494802,0.869006
...,...,...,...,...,...,...
14214,995,39,6.288155,-0.152285,0.999729,-0.023295
14215,996,39,6.289029,-0.110332,0.999869,-0.016198
14216,997,39,6.289636,-0.067059,0.999960,-0.008916
14217,998,39,6.289933,-0.027789,0.999994,-0.003433


In [372]:

df[['R','T']] = df.apply(lambda r: cart2pol(r['X'], r['Y']), axis=1, result_type='expand')
df['TD']      = df.apply(lambda r: rad2deg(r['T']), axis=1)

df['Step']    = df.apply(lambda r: f"{r.step}",  axis = 1) #  Circumference: {r.circ}",
df['face']    = df.apply(lambda r: f"i={r.I} | {round(r.TD)}°", axis = 1)

In [373]:
df

,I,step,X,Y,Nx,Ny,R,T,TD,Step,face
0,0,0,2.000000,0.000000,0.527437,0.849594,2.000000,0.000000,0.000000,0.0,i=0 | 0°
1,1,0,1.979941,0.012453,0.522855,0.852421,1.979980,0.006289,0.360360,0.0,i=1 | 0°
2,2,0,1.959805,0.024654,0.513586,0.858038,1.959960,0.012579,0.720721,0.0,i=2 | 1°
3,3,0,1.939595,0.036601,0.504235,0.863566,1.939940,0.018868,1.081081,0.0,i=3 | 1°
4,4,0,1.919312,0.048296,0.494802,0.869006,1.919920,0.025158,1.441441,0.0,i=4 | 1°
...,...,...,...,...,...,...,...,...,...,...,...
14214,995,39,6.288155,-0.152285,0.999729,-0.023295,6.289998,-0.024213,-1.387308,39.0,i=995 | -1°
14215,996,39,6.289029,-0.110332,0.999869,-0.016198,6.289997,-0.017542,-1.005069,39.0,i=996 | -1°
14216,997,39,6.289636,-0.067059,0.999960,-0.008916,6.289994,-0.010661,-0.610854,39.0,i=997 | -1°
14217,998,39,6.289933,-0.027789,0.999994,-0.003433,6.289995,-0.004418,-0.253134,39.0,i=998 | 0°


## Interactive Polar plot

In [374]:
Rub = np.ceil(df.R.max()) + 1


In [375]:

fig = px.line_polar(df, r="R", theta="TD", line_close=True,
                    range_r=[0,Rub], animation_frame="Step", 
                    direction= "counterclockwise", start_angle=0,
                    #color_discrete_sequence=px.colors.sequential.Plasma_r, 
                    #template="plotly_dark",)
                    width=600, height=600
                    )
fig.show()

In [ ]:
px.line(y = df.circ,x = df.step, width=600, height=600)

## More detailed plot

In [ ]:
def paint(id, palt):
    n = int(id%len(palt))
    return palt[n]

In [ ]:
extras = [pd.DataFrame(d) for d in extra]
extras = pd.concat(extras, ignore_index=True)
extras['lbl']     = extras.apply(lambda r: paint(r.step,pallettes.Prism), axis=1)
extras

In [ ]:
def decimate(df, upper_lim = 20000, factor = None):
    # randomly sample len(df)/factor data points from.
    # useful for plotting results with high amount of points 
    if df.shape[0] > upper_lim:
        if factor is None:
            factor = int(df.shape[0] / upper_lim) + 1
        print("df length:", df.shape[0] , "> upper limit:", upper_lim, "decimating by factor:", factor)

        import random
        rids = random.sample(sorted(df.index.values), int(df.shape[0]/factor))
        rids.sort()
        sset = df.loc[rids,:]
        return sset.reset_index()
    
    return df

dec_extras = decimate(extras)


In [ ]:
# plot only specific steps 

stepsToPlot = [2,3,4] #[2,3,4] # [8,9,10] #

slc = np.isin(df.step, stepsToPlot ) # range(10)

# plot only region of interest
roi = {'x':(None,None), 'y':(None,None)} # auto
#roi = {'x':(-16,16), 'y':(-16,16)} # whole shape
#roi = {'x':(0.5, 2.), 'y':(4.5, 6.5)} # one corner
#roi = {'x':(-2.5 , -4.0), 'y':(3.5 , 4.5)} # other corner


cols ="I, step, X, Y, Nx, Ny, face".split(', ')

# print(k, v.shape ), Px, Py, ti, tj, lbl

i, s, Xx, Yy, Nx, Ny, face = df.loc[slc,cols].T.values

In [ ]:
slc = np.isin(extras.step,  stepsToPlot) # range(10)

cols = "Px, Py, ti, tj, step, lbl".split(', ')

Px, Py, ti, tj, s, lbl = extras.loc[slc,cols].T.values

In [ ]:
fig = ff.create_quiver(Xx, Yy,  d * Nx,  d * Ny,
#fig = ff.create_quiver(X, Y,  Nx,  Ny,
                        #x, y, u, v,
                       scale=1, 
                       arrow_scale=.05,
                       name='offset',
                       line_width=2, 
                       angle=np.pi/6,
                     #  width=800, height=800)
)

fig.add_trace(go.Scatter (x=Xx, y=Yy,mode='lines', name='curves', text = face))
fig.add_trace(go.Scatter (x=Px, y=Py,mode='markers',
                           marker_color=lbl, 
                           name='intersections'))

fig.update_layout( width=800, height=800)
fig.update_xaxes(range=roi['x'])
fig.update_yaxes(range=roi['y'])
# Add points to figure
fig.show()

In [ ]:
res = df.loc[slc].to_dict()
extraslc = np.isin(extras.step,  stepsToPlot) # range(10)

extra = extras.loc[extraslc].to_dict()

In [ ]:
tmp

In [ ]:
tmp = res
tmp.update(extra)
dots_and_arrows(**tmp, d = d)

The new approach proves to be much more robust.

Though problems still exist. 
As I cut the intersecting points, the sampling resolution in the corners decreases, making the line more jagged.
This adversely affects the calculation of normals at next step, specifically in the corners, where accuracy is most crucial.
As a result, some normals may diverge significantly from proper direction, becoming a seed for a growing "swallow tail" artefact down the line.  

For now, an increase in sim resolution and shorter step can deal with this problem, but more robust methods should be used in the future.  

For example:
- Perform curve fitting and resampling of the corners points after each cut.
- Use some of the closest intersection points in the resulting curve.
- Transfer some of the corner points of original curve to the next one, like changing a hat.

# End